# M4 · 본문 추출과 청킹

앞 노트북에서 확인한 사례 6,032건과 공식 PDF 21개를 읽어 검색 가능한 조각으로 만든다. PDF는 페이지 번호를 붙이고, 텍스트가 거의 없는 페이지는 건너뛰되 몇 쪽인지 기록한다.

In [ ]:
from pathlib import Path
import json,hashlib
import pandas as pd
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None,'SANUP-P 프로젝트 폴더에서 실행하세요.'
PERSONAL=ROOT/'data'/'personal'
CORPUS=PERSONAL/'corpus';CORPUS.mkdir(parents=True,exist_ok=True)
REPORTS=PERSONAL/'reports';REPORTS.mkdir(parents=True,exist_ok=True)

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 1. 문서 파일 다시 확인

In [ ]:
docs_path=CORPUS/'documents.jsonl'
assert docs_path.exists(),'07 문서 목록 노트북을 먼저 실행하세요.'
documents=[json.loads(line) for line in docs_path.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(documents)==6053 and len({d['doc_id'] for d in documents})==6053
splitter=RecursiveCharacterTextSplitter(chunk_size=750,chunk_overlap=100,
     separators=['\n\n','\n','다. ',' ', ''])
print('문서 수',len(documents))

## 2. SIF 사례·PDF 페이지를 청크로 나누기

In [ ]:
chunk_path=CORPUS/'chunks.jsonl'
ocr_dir=CORPUS/'ocr_pages'
counts={};empty_pages=[];original_text_poor_pages=[];ocr_pages_added=[];chunk_ids=set();written=0
applied_text_corrections=[]
visual_text_corrections={
    ('A-G-4-2025',7): (
        '안전모를 착용하되 작업 높이가 미터 이상인 경우에는 안전모와 안전대를 함께 착용할 것  6. , 2',
        '안전모를 착용하되, 작업 높이가 2미터 이상인 경우에는 안전모와 안전대를 함께 착용할 것  6.',
    ),
}
with chunk_path.open('w',encoding='utf-8') as out:
    for doc in documents:
        if doc['source_type']=='sif':
            parts=[(None,doc['content'],'source_text')]
        else:
            path=ROOT/doc['local_path']
            assert path.is_file()
            reader=PdfReader(str(path))
            parts=[]
            for page_no,page in enumerate(reader.pages,1):
                text=(page.extract_text() or '').strip()
                method='pdf_text'
                correction=visual_text_corrections.get((doc['doc_id'],page_no))
                if correction:
                    original,corrected=correction
                    if text.count(original)!=1:
                        raise ValueError(f'원문 시각 교정 대상을 정확히 찾지 못함: {doc["doc_id"]} p{page_no}')
                    text=text.replace(original,corrected,1)
                    method='pdf_text_visual_correction'
                    applied_text_corrections.append({'doc_id':doc['doc_id'],'page':page_no,
                                                     'note':'PDF 원문 시각 확인으로 누락된 2미터 복구'})
                if len(text)<80:
                    original_text_poor_pages.append({'doc_id':doc['doc_id'],'page':page_no,'chars':len(text)})
                    ocr_path=ocr_dir/f"{doc['doc_id']}_p{page_no:03d}.json"
                    if ocr_path.exists():
                        ocr=json.loads(ocr_path.read_text(encoding='utf-8'))
                        if ocr['doc_id']==doc['doc_id'] and ocr['page']==page_no and ocr['accepted_for_index']:
                            text=ocr['text'];method=ocr['extraction_method']
                            ocr_pages_added.append({'doc_id':doc['doc_id'],'page':page_no,'chars':len(text)})
                if len(text)<80:
                    empty_pages.append({'doc_id':doc['doc_id'],'page':page_no,'chars':len(text)})
                    continue
                parts.append((page_no,text,method))
        for page_no,text,method in parts:
            for index,piece in enumerate(splitter.split_text(text)):
                if len(piece.strip())<40: continue
                chunk_id=hashlib.sha256(f"{doc['doc_id']}|{page_no}|{index}|{piece}".encode()).hexdigest()[:20]
                assert chunk_id not in chunk_ids
                chunk_ids.add(chunk_id)
                item={k:doc[k] for k in ('doc_id','source_id','source_type','title','organization',
                                         'source_url','published_at','license','industry_major','equipment')}
                item.update({'chunk_id':chunk_id,'page':page_no,'text':piece,
                             'extraction_method':method,'ocr_review_required':method.startswith('rapidocr')})
                out.write(json.dumps(item,ensure_ascii=False)+'\n')
                written+=1
                counts[doc['source_type']]=counts.get(doc['source_type'],0)+1
print('청크',written,'자료별',counts,'OCR 포함 페이지',len(ocr_pages_added),'검색 제외 페이지',len(empty_pages))


## 3. 추출 품질 기록

In [ ]:
report={'documents':len(documents),'chunks':written,'by_source_type':counts,
        'text_poor_pages':empty_pages,'original_text_poor_pages':original_text_poor_pages,
        'ocr_pages_added':ocr_pages_added,'manual_text_corrections':applied_text_corrections,
        'chunk_size':750,'overlap':100,
        'rules':['원본 문서 ID 보존','PDF 페이지 번호 보존','80자 미만 PDF는 검증된 OCR 결과를 우선 사용',
                 '숫자·단위는 필요한 경우 PDF 원문 화면과 대조','OCR 결과는 사람이 원문과 대조해야 함',
                 '40자 미만 청크 제외','원본 PDF 변경 없음']}
report_path=REPORTS/'rag_chunking.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('청크',chunk_path,'품질 기록',report_path)


## 청킹 결과와 품질

- 원본 문서 6,053개를 검색용 청크 6,845개로 나눴다. 자료 유형별 청크는 SIF 6,033개, 사고조사보고서 31개, KOSHA GUIDE 781개다. SIF 문서 중 일부는 길이가 길어 2개 이상 청크가 될 수 있다.
- PDF 578쪽 중 기존 추출 글자 수가 부족했던 105쪽을 OCR 점검 대상으로 잡았고, 99쪽의 텍스트를 복구해 최종적으로 OCR 표시 청크 114개를 만들었다. 나머지 6쪽은 무리하게 비어 있는 내용을 만들지 않고 제외했다.
- 각 청크에 문서 ID, 자료 유형, 제목, 기관, 출처 URL, 쪽, 업종·기인물 필터와 추출 방법을 보존했다. 청크 ID는 고유하며 텍스트·제목·URL 누락이 없는지 점검했다.
- 청크 경계는 검색 편의에 맞춘 분할이다. 원문 문맥이 잘리거나 표의 행·열 관계가 분리될 수 있으므로 검색 결과를 원문 전체의 의미로 간주하지 않는다.

### 이후 적재와 사용

- M4 임베딩은 코퍼스 해시와 색인 수를 맞춰야 하며, 청크가 바뀌면 이전 색인을 그대로 재사용하지 않는다.
- OCR 표시가 있는 문장은 답변 출처에 포함되면 PDF 해당 쪽을 확인한다.
- ①·⑥의 출처표시·변경금지 조건과 공개 사용 범위는 별도 확인 대상으로 남긴다.


## OCR 저텍스트 페이지 재확인 — 2026-10-03

- OCR 뒤에도 80자 미만인 6쪽을 원문과 대조했다. 5쪽은 PDF 표지이고, M-155-2023 20쪽은 ‘일일 점검표’ 구분 페이지다.
- 실제 점검표는 다음 PDF 쪽인 21쪽(인쇄 쪽수 18)에 있고 청크·색인에 포함되어 있다. ‘운전자격 적정여부 유도자 신호수 배치 안전인증’ 질의를 KOSHA GUIDE 문자 검색으로 실행했을 때 해당 페이지가 3위로 반환됐다.
- 본문이 빠진 페이지는 확인되지 않았다. 저텍스트 표지·구분 페이지만 제외해 검색 잡음을 줄였다. 상세 목록과 검증 질의는 data/personal/reports/ocr_excluded_page_review.md에 기록했다.
- 남은 주의사항: OCR 표시가 있는 답변은 PDF 원문에서 수치·단위를 대조한다. PDF 물리 쪽수와 인쇄 쪽수가 다를 수 있으므로 인용 시 둘 다 확인한다.
